# TTM FC01 — Zero-step integrated readiness
**Không train / không inference / không optimizer steps.** Đây là notebook kiểm tra quyền nguồn được owner xác nhận, Colab T4 FP32 và khả năng lưu–đọc lại trạng thái model/optimizer/RNG/cursor trên Google Drive. Không có lệnh mở D1/TEST hoặc chạy ba nhánh.

**Chuẩn bị một lần:** Trong Google Drive `MyDrive/TTM/ttm-small-pilot/ttm_small4m_fc01_fact_conditioning_20261113/input/`, upload 7 tệp private/source từ Windows đã pin: `FC01_OWNER_PRIVATE_DATA.json`, `T4_PRIVATE_TRAIN_VAL_WINDOWS.npz`, `T4_PRIVATE_PACK_MANIFEST.json`, `CF1_STEP1024_PINNED.pt`, `T1_BPE2048_REAL_TRAIN_MANIFEST.json`, `p02_exact.py`, `fc01_train_eval_once.py`. Hai file code cuối từ `_tmp/TTM_FC01_SOURCE_BUNDLE_20261113_v2/p02_exact.py` và `_tmp/fc01_onepacket_20261113_v2/fc01_train_eval_once.py`.

Chọn **Runtime → Change runtime type → T4 GPU**, chỉ dùng Colab Free. Đọc [CDLA-Sharing-1.0](https://cdla.dev/sharing-1-0/) và [TinyStories](https://huggingface.co/datasets/roneneldan/TinyStories). Sau đó đặt hai biến `OWNER_*` thành `True` nếu thực sự đồng ý. Không nhập dữ liệu riêng tư vào notebook công khai.

Kết quả và SHA nằm trong thư mục riêng `preexec_receipts/`. Bản snapshot step0 chỉ là **preflight**, không được đưa vào `train --resume`; nó không chứng minh resume thực tế tại step64.


In [ ]:
# FC01 T4 / FP32 / Drive one-packet preflight — strictly ZERO optimizer steps.
# Run interactively in Google Colab, free T4 preferred. NO model inference.
# Make an explicit personal owner decision after reviewing CDLA-Sharing-1.0.
OWNER_RIGHTS_ACCEPTED = False      # Set True ONLY after reviewing license and attribution.
OWNER_FREE_T4_COST_CAP_ACCEPTED = False  # Set True ONLY if using free T4, paid cost $0.

import os, sys, json, time, hashlib, shutil, random, importlib.util, traceback
from pathlib import Path

ROOT = Path("/content/drive/MyDrive/TTM/ttm-small-pilot/ttm_small4m_fc01_fact_conditioning_20261113")
INPUT = ROOT / "input"
OUT = ROOT / "preexec_receipts"
EXPECTED = {
 "FC01_OWNER_PRIVATE_DATA.json":"becc0408880aaed8bd5eb6fdf3997e62d5131c187a48331a03ae37d41aaa1aa3",
 "T4_PRIVATE_TRAIN_VAL_WINDOWS.npz":"184f68b4c0c800ef3e98362c2e20e1cd35baa269e926489a475664526ecd9913",
 "T4_PRIVATE_PACK_MANIFEST.json":"ef2e6e67d49b1d5a921b1051df329f15574f1c10cece0ce49468acb53322d650",
 "CF1_STEP1024_PINNED.pt":"cb985751795c8328ad65acf7ca4c1033ce35182c1bfbdb8c38550de629cb271b",
 "T1_BPE2048_REAL_TRAIN_MANIFEST.json":"cba18e60cee3d08a353ee6d4a451017289e45af7b4b1179c87171b9b78d7dfdc",
 "fc01_train_eval_once.py":"b786a4468ba104786df7683da9b1cf19d2334d9cc203dd19c3e7d6f10e2e93be",
}
P02_GIT_BLOB="9762343e06cf800b50574c2871ae1ed58537fb18"
CONFIG={"vocab_size":2048,"max_seq_len":256,"n_embd":256,"n_layer":4,"n_head":4,"norm_type":"layernorm"}
CORPUS_SHA=EXPECTED["FC01_OWNER_PRIVATE_DATA.json"]
STEP1024_SHA=EXPECTED["CF1_STEP1024_PINNED.pt"]

def sha256(path):
    h=hashlib.sha256()
    with open(path,"rb") as f:
        for chunk in iter(lambda:f.read(1024*1024),b""): h.update(chunk)
    return h.hexdigest()
def git_blob(raw):
    return hashlib.sha1(b"blob "+str(len(raw)).encode("ascii")+b"\0"+raw).hexdigest()
def tensor_hash(tensor):
    import torch
    t=tensor.detach().cpu().contiguous()
    return hashlib.sha256(t.numpy().tobytes()).hexdigest()
def fail(message): raise RuntimeError(message)
now=time.strftime("%Y%m%dT%H%M%SZ",time.gmtime())
status={
 "schema":"ttm.fc01.t4.fp32.drive.zero_step.preexec.v1",
 "operation":"TTM_SMALL4M_FC01_T4_FP32_DRIVE_RESUME_RIGHTS_SINGLE_INTEGRATED_SCIENCE_RELEASE_GO_HOLD_ZERO_TRAIN_STEPS",
 "run_utc":now,"science_optimizer_steps":0,"training":"NO","new_model_inference":"NO",
 "decoder_generation":"NO","test_q1q2":"SEALED","d1_confirmatory":"HOLD",
 "dev32":"CLOSED_UNCHANGED","source_hashes":{},
 "owner_rights_acknowledged":bool(OWNER_RIGHTS_ACCEPTED),
 "owner_free_t4_cost_cap_acknowledged":bool(OWNER_FREE_T4_COST_CAP_ACCEPTED),
 "license":"CDLA-Sharing-1.0","upstream":"https://huggingface.co/datasets/roneneldan/TinyStories",
 "license_text":"https://cdla.dev/sharing-1-0/",
 "rights_disclaimer":"Self-attested by owner in notebook. Not a third-party legal opinion.",
 "checkpoint_roundtrip":"NOT_EXECUTED",
 "real_runner_step64_resume":"NOT_EXECUTED_ZERO_STEPS_ONLY",
 "science_go":"NOT_GRANTED_BY_ZERO_STEP_PREFLIGHT",
 "sci_total_gpu_seconds_cap":10800,"sci_paid_USD_cap":0,
 "fp32":"NOT_EXECUTED",
 "status":"HOLD_PREEXEC"}
mounted=False
try:
    from google.colab import drive
    drive.mount("/content/drive",force_remount=False)
    mounted=True
    if not ROOT.is_dir() or not INPUT.is_dir() or not OUT.is_dir():
        fail("FC01_DRIVE_FOLDERS_MISSING")
    if not OWNER_RIGHTS_ACCEPTED: fail("FC01_OWNER_RIGHTS_NOT_ACCEPTED_NO_ASSUMPTION")
    if not OWNER_FREE_T4_COST_CAP_ACCEPTED: fail("FC01_OWNER_FREE_T4_COST_NOT_ACCEPTED")
    # Verify the exact six private scientific inputs and original P02 blob.
    for name,expected in EXPECTED.items():
        p=INPUT/name
        if not p.is_file() or p.is_symlink(): fail("FC01_INPUT_MISSING:"+name)
        got=sha256(p)
        status["source_hashes"][name]=got
        if got!=expected: fail("FC01_INPUT_SHA_MISMATCH:"+name+":"+got)
    p02=INPUT/"p02_exact.py"
    if not p02.is_file() or p02.is_symlink(): fail("FC01_P02_SOURCE_MISSING")
    p02_raw=p02.read_bytes()
    if git_blob(p02_raw)!=P02_GIT_BLOB: fail("FC01_P02_GIT_BLOB_MISMATCH")
    status["p02_git_blob"]=P02_GIT_BLOB
    cfg=json.loads((INPUT/"T4_PRIVATE_PACK_MANIFEST.json").read_text(encoding="utf8"))
    if cfg.get("contains_test") is not False: fail("FC01_PRIVATE_PACK_MUST_BE_TRAIN_VAL_ONLY")
    if cfg.get("npz_sha256")!=EXPECTED["T4_PRIVATE_TRAIN_VAL_WINDOWS.npz"]:
        fail("FC01_TRAIN_VAL_PACK_MANIFEST_SHA_MISMATCH")
    if cfg.get("t1_bpe_file_sha256")!=EXPECTED["T1_BPE2048_REAL_TRAIN_MANIFEST.json"]:
        fail("FC01_BPE_MANIFEST_NOT_PINNED")
    import torch,numpy as np
    status["torch_version"]=torch.__version__
    status["cuda_available"]=bool(torch.cuda.is_available())
    if not torch.cuda.is_available(): fail("FC01_COLAB_CUDA_UNAVAILABLE")
    gpu=torch.cuda.get_device_name(0)
    status["cuda_device_name"]=gpu
    if "T4" not in gpu.upper(): fail("FC01_NOT_NVIDIA_T4:"+gpu)
    torch.backends.cuda.matmul.allow_tf32=False
    torch.backends.cudnn.allow_tf32=False
    torch.use_deterministic_algorithms(True)
    torch.manual_seed(20261112)
    torch.cuda.manual_seed_all(20261112)
    random.seed(20261112)
    gen=np.random.default_rng(20261112)
    status["fp32"]="CUDA_T4_FP32_CONFIGURED_TF32_DISABLED"
    spec=importlib.util.spec_from_file_location("fc01_exact_p02_readiness",str(p02))
    mod=importlib.util.module_from_spec(spec)
    sys.modules[spec.name]=mod
    spec.loader.exec_module(mod)
    parent=torch.load(INPUT/"CF1_STEP1024_PINNED.pt",map_location="cpu",weights_only=True)
    if parent.get("step")!=1024 or parent.get("p02_git_blob")!=P02_GIT_BLOB or parent.get("config")!=CONFIG:
        fail("FC01_PARENT_CHECKPOINT_SCHEMA_CONFIG_DIFFERENT")
    model=mod.GPT(mod.P02Config(**CONFIG)).to("cuda",dtype=torch.float32)
    model.load_state_dict(parent["model"],strict=True)
    if sum(p.numel() for p in model.parameters())!=3749376:
        fail("FC01_P02_PARAMETER_COUNT_DIFFERENT")
    if any(p.dtype!=torch.float32 for p in model.parameters()):
        fail("FC01_MODEL_NOT_FP32")
    optimizer=torch.optim.AdamW(model.parameters(),lr=1e-4,betas=(0.9,0.95),weight_decay=0.1)
    if optimizer.state: fail("FC01_ZERO_STEP_OPTIMIZER_ALREADY_HAS_STATE")
    before={k:tensor_hash(v) for k,v in model.state_dict().items()}
    torch.cuda.synchronize()
    status["cuda_model_allocated_bytes"]=int(torch.cuda.memory_allocated(0))
    snapshot={
        "schema":"ttm.fc01.zero_step.preflight.snapshot.NOT_SCIENCE_CHECKPOINT.v1",
        "explicitly_not_valid_for_train_resume":True,
        "optimizer_updates_so_far":0,
        "parent_sha256":STEP1024_SHA,
        "corpus_sha256":CORPUS_SHA,
        "p02_git_blob":P02_GIT_BLOB,
        "arm":"UNASSIGNED_PREEXEC",
        "step":0,
        "model":model.state_dict(),
        "optimizer":optimizer.state_dict(),
        "torch_cpu_rng":torch.get_rng_state(),
        "torch_cuda_rng":torch.cuda.get_rng_state_all(),
        "python_rng":random.getstate(),
        "numpy_rng_state":gen.bit_generator.state,
        "core_cursor":0,"candidate_cursor":0,"synth_cursor":0,
    }
    # No forward, no backward, no optimizer.step; a pure save/load resume component smoke.
    local=Path("/content/fc01_zero_step_preflight_snapshot.pt")
    if local.exists(): fail("FC01_LOCAL_PREEXEC_SNAPSHOT_ALREADY_EXISTS_NO_OVERWRITE")
    torch.save(snapshot,local)
    local_sha=sha256(local)
    dest=OUT/("FC01_ZERO_STEP_DURABILITY_"+now+".pt")
    partial=Path(str(dest)+".part")
    if dest.exists() or partial.exists(): fail("FC01_DRIVE_OUTPUT_EXISTS_NO_OVERWRITE")
    with open(local,"rb") as inp, open(partial,"xb") as out:
        shutil.copyfileobj(inp,out,1024*1024)
        out.flush();os.fsync(out.fileno())
    if sha256(partial)!=local_sha: fail("FC01_DRIVE_PARTIAL_SHA_MISMATCH")
    os.replace(partial,dest)
    if sha256(dest)!=local_sha: fail("FC01_DRIVE_READBACK_SHA_MISMATCH")
    resumed=torch.load(dest,map_location="cpu",weights_only=True)
    if resumed.get("schema")!=snapshot["schema"] or resumed.get("optimizer_updates_so_far")!=0:
        fail("FC01_RESUME_SNAPSHOT_WRONG_SCHEMA_OR_NONZERO_STEP")
    if (resumed.get("parent_sha256")!=STEP1024_SHA or resumed.get("corpus_sha256")!=CORPUS_SHA
            or resumed.get("core_cursor")!=0 or resumed.get("candidate_cursor")!=0
            or resumed.get("synth_cursor")!=0):
        fail("FC01_RESUME_SNAPSHOT_SOURCE_OR_CURSOR_MISMATCH")
    clone=mod.GPT(mod.P02Config(**CONFIG)).to("cuda",dtype=torch.float32)
    clone.load_state_dict(resumed["model"],strict=True)
    clone_optimizer=torch.optim.AdamW(clone.parameters(),lr=1e-4,betas=(0.9,0.95),weight_decay=0.1)
    clone_optimizer.load_state_dict(resumed["optimizer"])
    if clone_optimizer.state: fail("FC01_ZERO_STEP_OPTIMIZER_STATE_MUTATED")
    restored={k:tensor_hash(v) for k,v in clone.state_dict().items()}
    if before!=restored: fail("FC01_RESUME_MODEL_TENSOR_MISMATCH")
    torch.set_rng_state(resumed["torch_cpu_rng"].cpu())
    torch.cuda.set_rng_state_all(resumed["torch_cuda_rng"])
    random.setstate(resumed["python_rng"])
    gen2=np.random.default_rng()
    gen2.bit_generator.state=resumed["numpy_rng_state"]
    if gen2.bit_generator.state!=gen.bit_generator.state:
        fail("FC01_NUMPY_SAMPLER_RNG_STATE_MISMATCH")
    if not torch.equal(torch.get_rng_state(),resumed["torch_cpu_rng"]):
        fail("FC01_CPU_RNG_RESTORE_FAIL")
    if not all(torch.equal(a,b) for a,b in zip(torch.cuda.get_rng_state_all(),resumed["torch_cuda_rng"])):
        fail("FC01_CUDA_RNG_RESTORE_FAIL")
    torch.cuda.synchronize()
    status.update({
       "checkpoint_roundtrip":"PASS_ZERO_STEP_MODEL_OPTIMIZER_RNG_CURSOR_DRIVE",
       "preflight_snapshot_filename":dest.name,
       "preflight_snapshot_sha256":local_sha,
       "preflight_snapshot_bytes":int(dest.stat().st_size),
       "input_hashes_verified":len(EXPECTED)+1,
       "t4_factual_available_in_session":True,
       "resume_limit":"STEP0_QA_SNAPSHOT_IS_NOT_A_REAL_FC01_TRAINING_CHECKPOINT; STEP64_TRAIN_RESUME_NOT_TESTED",
       "science_optimizer_steps":0,
       "status":"PASS_COLAB_ZERO_STEP_SCOPED_ONLY_NOT_SCIENCE_GO",
    })
except Exception as exc:
    status["status"]="HOLD_PREEXEC"
    status["blocker"]=type(exc).__name__+":"+str(exc)
    print("FC01_PREFLIGHT_HOLD:",status["blocker"])
finally:
    status["sci_paid_USD_observed"]="NOT_QUERYABLE_FROM_COLAB_RUNTIME; OWNER_ACK_REQUIRED"
    if mounted and OUT.is_dir():
        receipt=OUT/("FC01_T4_FP32_DRIVE_ZERO_STEP_READINESS_"+now+".json")
        try:
            with receipt.open("x",encoding="utf8") as f:
                json.dump(status,f,ensure_ascii=True,indent=2,sort_keys=True)
                f.write("\n");f.flush();os.fsync(f.fileno())
            print("FC01_RECEIPT_PATH:",str(receipt))
            print("FC01_RECEIPT_SHA256:",sha256(receipt))
        except Exception as subexc:
            print("FC01_RECEIPT_WRITE_FAILED:",type(subexc).__name__,str(subexc))
    print("FC01_FINAL_READINESS_STATUS:",status["status"])
    print("FC01_SCIENCE_RELEASE: NOT_GRANTED_BY_ZERO_STEP_NOTEBOOK")
    print("FC01_OPTIMIZER_STEPS:",status["science_optimizer_steps"])
